In [ ]:
def plot_generation_versus_demand(region: int, events_data_input_dir: str, metadata_input_dir: str, image_output_dir: str, image_resolution: int, save_images=False):

    # Read in NERC region name file and extract the TPL-08 region name:
    nerc = pd.read_csv((metadata_input_dir + 'nerc_tpl08_region_names.csv'))
    nerc_name = nerc.loc[nerc['short_name'] == region, 'long_name'].item()
           
    # Extract the heat wave library data for a given NERC region:
    hw_df = pd.read_csv((events_data_input_dir + 'hw_library_expanded.csv'))

    # Calculate the implied total demand at the time of peak load:
    hw_df['Inferred_Peak_Demand'] = hw_df['Total_at_Peak'] + hw_df['Import_at_Peak'] + hw_df['Load_Shed_at_Peak']
    
    # Subset to just the data for region you want to use and drop rows containing NaN load values (i.e., no GridView results):
    hw_subset_df = hw_df[(hw_df['Region'] == region)].copy()
    hw_subset_df.dropna(subset=['Inferred_Peak_Demand'], inplace=True)
    
    # Extract the cold snap library data for a given NERC region:
    cs_df = pd.read_csv((events_data_input_dir + 'cs_library_expanded.csv'))

    # Calculate the implied total demand at the time of peak load:
    cs_df['Inferred_Peak_Demand'] = cs_df['Total_at_Peak'] + cs_df['Import_at_Peak'] + cs_df['Load_Shed_at_Peak']
    
    # Subset to just the data for region you want to use and drop rows containing NaN load values (i.e., no GridView results):
    cs_subset_df = cs_df[(cs_df['Region'] == region)].copy()
    cs_subset_df.dropna(subset=['Inferred_Peak_Demand'], inplace=True)

    # Calculate the minimum and maximum load values to use in plotting:
    load_min = min([hw_subset_df['Region_Load_Peak'].min(), hw_subset_df['Inferred_Peak_Demand'].min(), cs_subset_df['Region_Load_Peak'].min(), cs_subset_df['Inferred_Peak_Demand'].min()])*0.98
    load_max = max([hw_subset_df['Region_Load_Peak'].max(), hw_subset_df['Inferred_Peak_Demand'].max(), cs_subset_df['Region_Load_Peak'].max(), cs_subset_df['Inferred_Peak_Demand'].max()])*1.02
    
    # Create a 1:1 line for plotting:
    one_to_one = np.arange(load_min, load_max, ((load_max - load_min)/30))
    
    # Make the scatter plot:
    plt.figure(figsize=(15, 15))
    plt.rcParams['font.size'] = 18
    plt.rcParams['axes.axisbelow'] = True
    plt.plot(one_to_one,one_to_one,'gray', linewidth=2, label='1:1')
    plt.plot(one_to_one,one_to_one*1.1,'gray', linewidth=2, linestyle=':', label='1:1 +/-10%')
    plt.plot(one_to_one,one_to_one*0.9,'gray', linewidth=2, linestyle=':')
    plt.scatter(hw_subset_df['Region_Load_Peak'], hw_subset_df['Inferred_Peak_Demand'], s=35, c='r', label=('R$^2$=' + str(hw_subset_df['Region_Load_Peak'].corr(hw_subset_df['Inferred_Peak_Demand']).round(3))))
    plt.scatter(cs_subset_df['Region_Load_Peak'], cs_subset_df['Inferred_Peak_Demand'], s=35, c='b', label=('R$^2$=' + str(cs_subset_df['Region_Load_Peak'].corr(cs_subset_df['Inferred_Peak_Demand']).round(3))))
    plt.grid(linestyle='-', linewidth=0.5, color='gray')
    plt.legend(loc='upper left')
    plt.xlabel('Regional Peak Demand [MW]') 
    plt.ylabel('Implied Regional Peak Demand [Generation + Imports + Load Shedding]')
    plt.xlim(load_min, load_max)
    plt.ylim(load_min, load_max)
    plt.title('Demand vs Inferred Demand: ' + nerc_name + ' Heat Wave and Cold Snap Events')

    # If the "save_images" flag is set to true then save the plot to a .png file:
    if save_images == True:
       plt.savefig(os.path.join(image_output_dir + str(region) + '_Inferrred_Demand_Comparison.png'), dpi=image_resolution, bbox_inches='tight')
    

In [ ]:
# Test the function:
plot_generation_versus_demand(region = 'SW',
                              events_data_input_dir = events_data_input_dir,
                              metadata_input_dir = metadata_input_dir,
                              image_output_dir = image_output_dir, 
                              image_resolution = 150,
                              save_images = False)
